In [9]:
import lab

lab.init(target="local")  # this run is declared straight to the real volume

In [10]:
from artifacts.core.artifact import Resources
from artifacts.core.SGD.training import (
    LoopConfig,
    LRSchedule,
    OptimizerParameters,
    TrainingParameters,
)
from artifacts.dataset import DataSet
from artifacts.mappeddataset import MappedDataSet
from artifacts.models.transformer import ModelParameters, Pretraining
from artifacts.sources import Source
from artifacts.tokenizers.bpe import Tokenizer as BPETokenizer

odyssey = Source(name="odyssey", url="https://www.gutenberg.org/cache/epub/1727/pg1727.txt")
mobydick = Source(name="mobydick", url="https://www.gutenberg.org/cache/epub/2701/pg2701.txt")
romeojuliet = Source(name="romeojuliet", url="https://www.gutenberg.org/cache/epub/1513/pg1513.txt")
montecristo = Source(name="montecristo", url="https://www.gutenberg.org/cache/epub/1184/pg1184.txt")
pride = Source(name="pride", url="https://www.gutenberg.org/cache/epub/1342/pg1342.txt")
frankenstein = Source(name="frankenstein", url="https://www.gutenberg.org/cache/epub/84/pg84.txt")

RUN_ID = "MODEL_LEGS"  # <- change this per run

tokenizer = BPETokenizer(
    vocab_size=1000,
    special_tokens=("<pad>", "<unk>"),
    sources=(odyssey, mobydick),
)


justdataset = DataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[
        montecristo,
        mobydick,
        frankenstein,
    ],
    valid_sources=[romeojuliet, pride],
)


mappedset = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[
        montecristo,
        mobydick,
        frankenstein,
    ],
    valid_sources=[romeojuliet, pride],
)


mappedset2 = MappedDataSet.from_sources(
    tokenizer=tokenizer,
    train_sources=[
        montecristo,
        pride,
        frankenstein,
    ],
    valid_sources=[romeojuliet, pride],
)

optimizer_parameters = OptimizerParameters(
    lr= 0.001,
    betas= (0.9, 0.999),
    weight_decay = 0.1,
    eps = 1e-8,
)

model_parameters = ModelParameters(
    vocab_size=tokenizer.vocab_size,
    sequence_length=64,
    num_layers=1,
    d_model=64,
    d_ff=128,
    num_heads=4,
    rope_theta=10000,
    device="cuda",  # has to agree with app.GPU; the preflight checks it
    dtype="torch.float32", #strings parsed in the worker container.
)

lr_schedule = LRSchedule(
    max_learning_rate=1e-3,
    min_learning_rate=1e-4,
    warmup_iters=100,
    cosine_cycle_iters=10000,
)

training_parameters = TrainingParameters(
    total_steps=1000,
    batch_size=64,
    max_norm=1.0,
    lr_schedule=lr_schedule,
    optimizer="torch.optim.AdamW",
    optimizer_parameters=optimizer_parameters,
    seed=0,
)

loop_config = LoopConfig(
    checkpoint_every=500,
    val_every=5000,
    gpu_check_every=100,
    optimizer_checkpoint_policy="latest",
)

pretraining_0 = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset,
    tokenizer=tokenizer,
    model_parameters=model_parameters,
    training_parameters=training_parameters,
    loop_config=loop_config,
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

pretraining_1 = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset2,
    tokenizer=tokenizer,
    starting_checkpoint=pretraining_0,
    training_parameters=training_parameters,
    loop_config=loop_config,
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

pretraining_2 = Pretraining(
    run_id=RUN_ID,
    dataset=mappedset,
    tokenizer=tokenizer,
    starting_checkpoint=pretraining_1,
    training_parameters=training_parameters,
    loop_config=loop_config,
    allocated_resources=Resources(gpu_type="T4", gpu_count=1),
)

pretraining = pretraining_2

DEFINITION = "\n".join(In[-1].splitlines()[:-1])  # this cell's own source, minus this line

In [6]:
for artifact in lab.plan(justdataset):
    artifact.job().run(lab.root(), lab.worker)

downloading odyssey from https://www.gutenberg.org/cache/epub/1727/pg1727.txt
wrote 710363 chars for odyssey
downloading mobydick from https://www.gutenberg.org/cache/epub/2701/pg2701.txt
wrote 1260595 chars for mobydick
training BPE tokenizer (vocab_size=1000) on 2 source(s)
trained, vocab has 1000 entries
downloading montecristo from https://www.gutenberg.org/cache/epub/1184/pg1184.txt
wrote 2708329 chars for montecristo
tokenizing montecristo
wrote 1097739 tokens for montecristo
tokenizing mobydick
wrote 495382 tokens for mobydick
downloading frankenstein from https://www.gutenberg.org/cache/epub/84/pg84.txt
wrote 446582 chars for frankenstein
tokenizing frankenstein
wrote 174223 tokens for frankenstein
downloading romeojuliet from https://www.gutenberg.org/cache/epub/1513/pg1513.txt
wrote 167469 chars for romeojuliet
tokenizing romeojuliet
wrote 76295 tokens for romeojuliet
downloading pride from https://www.gutenberg.org/cache/epub/1342/pg1342.txt
wrote 763082 chars for pride
toke

In [7]:
from artifacts.core.resolve import resolve

In [ ]:
odyssey.declare()

Source(name='odyssey', url='https://www.gutenberg.org/cache/epub/1727/pg1727.txt')